# Try your formality model
Enter a sentence and get a formal rewrite using your saved T5-small model. **This notebook does not train anything.**

Run this locally in VS Code using the project’s `.venv` Python kernel. Keep the notebook and your model folder or ZIP in the project directory. No NVIDIA GPU is needed.

## 1. Set up your local environment
Dependencies are managed in `pyproject.toml`. From a terminal in the project directory, run:

```bash
uv sync
```

Then select **Select Kernel → Python Environments → .venv/bin/python** in VS Code. If it is not listed, use **Enter interpreter path** and choose that file. The project uses Python 3.12 by default. Run the cell below to confirm the selected interpreter. Installation needs internet; inference uses local model files.

In [ ]:
import sys
print("Python:", sys.executable)
print("Version:", sys.version.split()[0])

## 2. Choose your saved model
Set `MODEL_DIR` to the saved model folder you want to use. The default is `models/formalize-t5`, matching the `EXPORT_DIR` written by `train_formal.ipynb`.

Paths are resolved from the project root, found by walking up to `pyproject.toml`. That matters because VS Code starts the kernel in the **notebook's** folder by default, so a bare relative path like `models/...` would resolve to `notebooks/models/...` and fail.

If the chosen folder does not exist but its ZIP sits beside it, this cell extracts the ZIP — which requires the ZIP's top-level folder name to match `MODEL_DIR`'s name. An existing folder is never overwritten.

`MAX_INPUT_TOKENS` stays at 128 even though the model was trained with a 64-token limit. Inputs above the limit are **rejected rather than truncated**, so no text is ever silently dropped, and 128 avoids refusing text the model can still attempt. Section 6 shows what actually happens to long input.

In [ ]:
from pathlib import Path
import zipfile

# Find the project root by walking up to pyproject.toml, so the paths below work
# whichever directory the kernel starts in (VS Code defaults to the notebook's folder).
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
MODEL_DIR = PROJECT_ROOT / "models/formalize-t5"  # Matches EXPORT_DIR in train_formal.ipynb.
PREFIX = "formalize: "  # Must match the training prefix.
MAX_INPUT_TOKENS = 128
MAX_NEW_TOKENS = 128
NUM_BEAMS = 4

MODEL_DIR = MODEL_DIR.expanduser().resolve()
archive_path = MODEL_DIR.with_suffix(".zip")
if not MODEL_DIR.exists():
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Cannot find {MODEL_DIR} or {archive_path}. "
            "Choose the path to your extracted local model folder."
        )
    with zipfile.ZipFile(archive_path) as archive:
        # Check paths before extracting the model folder.
        for member in archive.infolist():
            destination = (MODEL_DIR.parent / member.filename).resolve()
            if not destination.is_relative_to(MODEL_DIR):
                raise ValueError(f"Unexpected ZIP member: {member.filename}")
        archive.extractall(MODEL_DIR.parent)
    print("Extracted:", archive_path.name)

required = ["config.json", "model.safetensors", "tokenizer_config.json", "tokenizer.json"]
missing = [name for name in required if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete model folder {MODEL_DIR}: missing {missing}")
print("Model folder:", MODEL_DIR)

## 3. Load the model once
Inference uses your local CPU. Loading happens once; subsequent sentences reuse the model. No NVIDIA GPU, Hugging Face login, or model download is needed.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = torch.device("cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR, local_files_only=True)
model.to(device)
model.eval()
print(f"Ready! Using {device}.")

## 4. Define the rewriting function
Use the same `formalize: ` prefix as training — it is a learned task token, so the model produces nonsense without the exact string. Beam search runs without random sampling. Blank input returns an empty string; inputs over 128 tokens are rejected so text is not silently dropped.

In [ ]:
@torch.inference_mode()
def formalize(text):
    """Rewrite one sentence with the saved model."""
    if not isinstance(text, str):
        raise TypeError("Please provide a string.")
    text = text.strip()
    if not text:
        return ""

    inputs = tokenizer(PREFIX + text, return_tensors="pt", truncation=False)
    if inputs["input_ids"].shape[1] > MAX_INPUT_TOKENS:
        raise ValueError("That text is too long. Please try a shorter sentence.")
    inputs = inputs.to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

## 5. Quick example
Run this to check that loading and generation work. A successful run verifies inference, not the quality of every rewrite.

In [ ]:
example = "gonna need that report asap, u totally dropped the ball on this one"
print("Original: ", example)
print("Formal:   ", formalize(example))

## 6. Probe the limits: long and unpunctuated input
GYAFC is a **sentence-level** corpus — its longest example is 58 tokens — so the model has no training signal for long passages, whatever `MAX_INPUT_TOKENS` is set to. Its informal side also comes from written Yahoo Answers text, which is punctuated, so unpunctuated voice-typed input is unfamiliar in a second way.

This cell makes both failure modes visible instead of leaving them to guesswork:

1. a normal short message — the case the model was trained for
2. a long message that still fits under the 128-token cap, so nothing is truncated on the way in; watch whether content goes missing on the way **out**
3. the same long message with punctuation stripped, simulating voice typing
4. a message over the 128-token cap, showing that oversized input is **rejected with an error** rather than quietly cut in half

Token counts are printed alongside, because content loss is much easier to see as a number than to spot by reading. Use what you observe here to decide whether a longer-input strategy is worth building — the likely candidate is punctuation restoration followed by sentence splitting, formalizing each sentence, then rejoining.

In [ ]:
import re

short_message = "hey can u send me the deck b4 the mtg, kinda urgent"
long_message = (
    "Hey so i wanted to flag that the deployment yesterday went pretty badly and honestly "
    "nobody on the team had any clue what was going on, the staging env was totally broken "
    "for like three hours and we lost a bunch of test data, plus the rollback script didnt "
    "even work so we had to do it all by hand which was a nightmare, anyway can we please "
    "talk about this in standup tomorrow because it cant happen again"
)
unpunctuated = re.sub(r"[^\w\s]", "", long_message).lower()
oversized = long_message + " " + long_message  # Comfortably past MAX_INPUT_TOKENS.

for label, text in [
    ("1. short (expected case)", short_message),
    ("2. long, punctuated", long_message),
    ("3. long, no punctuation", unpunctuated),
    ("4. over the token cap", oversized),
]:
    print("=" * 70)
    print(label)
    input_tokens = tokenizer(PREFIX + text, truncation=False)["input_ids"]
    try:
        rewritten = formalize(text)
    except ValueError as error:
        print(f"  input tokens: {len(input_tokens)}  -> rejected: {error}")
        continue
    output_tokens = tokenizer(rewritten, truncation=False)["input_ids"]
    print(f"  input  ({len(input_tokens):>3} tokens): {text}")
    print(f"  output ({len(output_tokens):>3} tokens): {rewritten}")
    print(f"  output/input token ratio: {len(output_tokens) / len(input_tokens):.2f}")
    print("  (a ratio well below 1.0 on a long input means content was dropped)")

## 7. Type your own sentence
Edit the string below and run the cell. Rerun **only this cell** for another sentence.

Alternatively, call `formalize("your sentence here")` in a new cell.

Review each result: the model may leave informal wording in place, change meaning, or correctly leave an already-formal sentence unchanged.

In [ ]:
sentence = "i dont think thats gonna work tbh"  # input("Enter a sentence to formalize: ")
if not sentence.strip():
    print("No sentence entered. Rerun this cell to try again.")
else:
    try:
        print("Original: ", sentence)
        print("Formal:   ", formalize(sentence))
    except ValueError as error:
        print(error)